# Figure 2 Notebook — Healthy MERFISH Dataset
## Cell Type Annotation, Spatial Analysis, and Vascular Characterization

**Dataset:** Healthy control MERFISH pancreas (samples 93, 17, 162 excluded due to QC)
**Input:** `healthycells_allsamples_no9317162_scvi.h5ad` (scVI pre-integrated)
**Output:** `healthycells_allsamples_no9317162_labeledCV.h5ad`

### Pipeline Overview
1. CONCORD batch integration (domain_key = Sample)
2. Leiden clustering + cell type annotation (broad + fine-grained)
3. DEG analysis per cell type → SI CSV export
4. Islet ROI geometry extraction from MERFISH boundary files
5. Spatial distance computation: cell → islet boundary (signed µm)
6. Cell → nearest endothelial, mesenchymal, and beta cell distances

### Key Output Columns in adataf.obs
| Column | Description |
|---|---|
| `concord_leiden_1` | Raw Leiden clusters (res=1.0) |
| `merged_cell_type` | Broad annotation (endocrine subtypes pooled) |
| `specific_cell_type` | Fine-grained (edge alpha, plasma/alpha retained) |
| `dist_edge_islet_um` | Signed distance to nearest islet boundary (µm); negative = inside |
| `dist_edge_endo_um` | Distance to nearest endothelial cell (µm) |
| `dist_edge_mesenchymal_um` | Distance to nearest mesenchymal cell (µm) |
| `dist_edge_beta_um` | Distance to nearest beta cell (µm) |

### Key Gene Lists
| Variable | Description |
|---|---|
| `marker_genes_dict` | Curated markers for all major pancreatic cell types |
| `vasc_marker_genes_dict` | Vascular/stromal-specific markers |
| `ECM` | Full ECM gene list (61 genes) |
| `ECM_dict` | ECM genes grouped: Collagens, Laminins, Others |

### Cluster → Cell Type Mapping (merged_cell_type)
| Leiden Clusters | Cell Type |
|---|---|
| 3, 7, 10, 9, 19 | Alpha Cells |
| 1, 11 | Beta Cells |
| 16, 17, 2 | Ductal Cells |
| 0, 4, 6 | Acinar Cells |
| 8, 13 | Endothelial Cells |
| 5, 14 | Mesenchymal Cells |
| 15 | Immune Cells |
| 12 | Delta Cells |
| 18 | PPY Cells |
| 20 | Removed (low quality) |

In [ ]:
import concord as ccd
import scanpy as sc
import torch
import os
import pandas as pd
import numpy as np
import anndata as ad

# SECTION 2 — DATA LOADING + CONCORD INTEGRATION
 Input: scVI-integrated h5ad (samples 93/17/162 excluded)
 normalize → log1p → CONCORD (domain_key=Sample) → UMAP
 Note: cosine metric, n_neighbors=30 chosen for sparse spatial data

In [ ]:
import concord as ccd
import scanpy as sc
import torch
import os
import pandas as pd
import numpy as np
import anndata as ad

In [ ]:
os.chdir('/Volumes/Samsung_4TB/Desktop_copy/Islet_exorine_andatas')
adata67_in = sc.read_h5ad('HuP67A_isletcells.h5ad')
adata03_in = sc.read_h5ad('HuP03A_isletcells.h5ad')
adata31_in = sc.read_h5ad('HuP31A_isletcells.h5ad')
adata20_in = sc.read_h5ad('HuP20A_isletcells.h5ad')
adata53_in = sc.read_h5ad('HuP53A_isletcells.h5ad')
adata156_in = sc.read_h5ad('HuP156A_isletcells.h5ad')
adata71_in = sc.read_h5ad('HuP71A_isletcells.h5ad')




In [ ]:
os.chdir('/Volumes/Samsung_4TB/Desktop_copy/Islet_exorine_andatas/')
adata67_out = sc.read_h5ad('HuP67A_NONisletcells.h5ad')
adata03_out = sc.read_h5ad('HuP03A_NONisletcells.h5ad')
adata31_out = sc.read_h5ad('HuP31A_NONisletcells.h5ad')
adata20_out = sc.read_h5ad('HuP20A_NONisletcells.h5ad')
adata53_out = sc.read_h5ad('HuP53A_NONisletcells.h5ad')
adata156_out = sc.read_h5ad('HuP156_NONisletcells.h5ad')
adata71_out = sc.read_h5ad('HuP71A_NONisletcells.h5ad')


In [ ]:
adata162_out

In [ ]:
adata03_in.obs['Location'] = 'Islet'
adata20_in.obs['Location'] = 'Islet'
adata31_in.obs['Location'] = 'Islet'
adata67_in.obs['Location'] = 'Islet'
adata53_in.obs['Location'] = 'Islet'
adata71_in.obs['Location'] = 'Islet'
adata156_in.obs['Location'] = 'Islet'



adata03_in.obs['Sample'] = 'HuP03A'
adata20_in.obs['Sample'] = 'HuP20A'
adata31_in.obs['Sample'] = 'HuP31A'
adata67_in.obs['Sample'] = 'HuP67A'

adata53_in.obs['Sample'] = 'HuP53A'
adata71_in.obs['Sample'] = 'HuP71A'
adata156_in.obs['Sample'] = 'RHIP156'


adata03_in.obs['Health_Status'] = 'ND'
adata20_in.obs['Health_Status'] = 'ND'
adata31_in.obs['Health_Status'] = 'ND'
adata67_in.obs['Health_Status'] = 'ND'

adata53_in.obs['Health_Status'] = 'ND'
adata71_in.obs['Health_Status'] = 'ND'
adata156_in.obs['Health_Status'] = 'ND'


adata03_in.obs['Sex'] = 'Male'
adata20_in.obs['Sex'] = 'Male'
adata31_in.obs['Sex'] = 'Female'
adata67_in.obs['Sex'] = 'Male'

adata53_in.obs['Sex'] = 'Female'
adata71_in.obs['Sex'] = 'Male'
adata156_in.obs['Sex'] = 'Female'


adata03_in.obs['Age'] = '75'
adata20_in.obs['Age'] = '46'
adata31_in.obs['Age'] = '73'
adata67_in.obs['Age'] = '70'

adata53_in.obs['Age'] = '41'
adata71_in.obs['Age'] = '48'
adata156_in.obs['Age'] = '48'


adata03_in.obs['BMI'] = '30.2'
adata20_in.obs['BMI'] = '26.9'
adata31_in.obs['BMI'] = '33.1'
adata67_in.obs['BMI'] = '29.3'

adata53_in.obs['BMI'] = '19.3'
adata71_in.obs['BMI'] = '29.4'
adata156_in.obs['BMI'] = '24.1'


adata03_in.obs['Ethnicity'] = 'Caucasian'
adata20_in.obs['Ethnicity'] = 'Caucasian'
adata31_in.obs['Ethnicity'] = 'Caucasian'

adata67_in.obs['Ethnicity'] = 'Caucasian'
adata53_in.obs['Ethnicity'] = 'Caucasian'
adata71_in.obs['Ethnicity'] = 'Caucasian'
adata156_in.obs['Ethnicity'] = 'African American'


In [ ]:
adata03_out.obs['Sample'] = 'HuP03A'
adata20_out.obs['Sample'] = 'HuP20A'
adata31_out.obs['Sample'] = 'HuP31A'
adata67_out.obs['Sample'] = 'HuP67A'
adata53_out.obs['Sample'] = 'HuP53A'

adata71_out.obs['Sample'] = 'HuP71A'
adata156_out.obs['Sample'] = 'RHIP156'


adata03_out.obs['Location'] = 'Exocrine'
adata20_out.obs['Location'] = 'Exocrine'
adata31_out.obs['Location'] = 'Exocrine'
adata67_out.obs['Location'] = 'Exocrine'

adata53_out.obs['Location'] = 'Exocrine'
adata71_out.obs['Location'] = 'Exocrine'
adata156_out.obs['Location'] = 'Exocrine'


adata03_out.obs['Health_Status'] = 'ND'
adata20_out.obs['Health_Status'] = 'ND'
adata31_out.obs['Health_Status'] = 'ND'
adata67_out.obs['Health_Status'] = 'ND'

adata53_out.obs['Health_Status'] = 'ND'
adata71_out.obs['Health_Status'] = 'ND'
adata156_out.obs['Health_Status'] = 'ND'


adata03_out.obs['Sex'] = 'Male'
adata20_out.obs['Sex'] = 'Male'
adata31_out.obs['Sex'] = 'Female'
adata67_out.obs['Sex'] = 'Male'

adata53_out.obs['Sex'] = 'Female'
adata71_out.obs['Sex'] = 'Male'
adata156_out.obs['Sex'] = 'Female'


adata03_out.obs['Age'] = '75'
adata20_out.obs['Age'] = '46'
adata31_out.obs['Age'] = '73'
adata67_out.obs['Age'] = '70'

adata53_out.obs['Age'] = '41'
adata71_out.obs['Age'] = '48'
adata156_out.obs['Age'] = '48'

adata03_out.obs['BMI'] = '30.2'
adata20_out.obs['BMI'] = '26.9'
adata31_out.obs['BMI'] = '33.1'
adata67_out.obs['BMI'] = '29.3'

adata53_out.obs['BMI'] = '19.3'
adata71_out.obs['BMI'] = '29.4'
adata156_out.obs['BMI'] = '24.1'


adata03_out.obs['Ethnicity'] = 'Caucasian'
adata20_out.obs['Ethnicity'] = 'Caucasian'
adata31_out.obs['Ethnicity'] = 'Caucasian'
adata67_out.obs['Ethnicity'] = 'Caucasian'

adata53_out.obs['Ethnicity'] = 'Caucasian'
adata71_out.obs['Ethnicity'] = 'Caucasian'
adata156_out.obs['Ethnicity'] = 'African American'



In [ ]:
healthy_datas = [adata03_in, adata20_in, adata31_in, adata67_in, adata53_in, adata156_in, adata71_in] 

In [ ]:
healthy_islet_cells = sc.concat(healthy_datas, index_unique='_')
healthy_islet_cells

In [ ]:
healthy_datas_exo = [adata03_out, adata20_out, adata31_out, adata67_out, adata53_out, adata156_out, adata71_out  ]

In [ ]:
healthy_exo_cells = sc.concat(healthy_datas_exo, index_unique='_')
healthy_exo_cells

In [ ]:
alldata= [healthy_islet_cells, healthy_exo_cells]

In [ ]:
alldata_cells = sc.concat(alldata, index_unique='_')
alldata_cells

In [ ]:
healthyexo_cells = sc.pp.subsample(healthy_exo_cells, fraction=0.025, copy=True)
healthyexo_cells

In [ ]:
alldata_2= [healthy_islet_cells, healthyexo_cells]

In [ ]:
adata = sc.concat(alldata_2, index_unique='_')
adata

In [ ]:
adata.write_h5ad('/Users/kmeneses/Desktop/updated_data/healthycells_allsamples_n93217.h5ad')

In [ ]:
adata = sc.read_h5ad('/Users/kmeneses/Desktop/updated_data/healthycells_allsamples_no9317162_scvi.h5ad')
adata

In [ ]:
adata.obs['Sample']

In [ ]:
sc.pp.normalize_total(adata) # Normalize counts per cell
sc.pp.log1p(adata)

In [ ]:
# Set device to cpu or to gpu (if your torch has been set up correctly to use GPU), for mac you can use either torch.device('mps') or torch.device('cpu')
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Initialize Concord with an AnnData object, skip input_feature to use all features, set preload_dense=False if your data is very large
# Provide 'domain_key' if integrating across batches, see below
cur_ccd = ccd.Concord(adata=adata, input_feature=None, domain_key='Sample', device=device, preload_dense=True) 

# Encode data, saving the latent embedding in adata.obsm['Concord']
cur_ccd.fit_transform(output_key='Concord')

In [ ]:
ccd.ul.run_umap(adata, source_key='Concord', result_key='Concord_UMAP', n_components=2, n_neighbors=30, min_dist=0.1, metric='cosine')

# Plot the UMAP embeddings
color_by = ['Sample', 'Location'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    adata, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data'
)

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

# =========================
# PLOT
# =========================
sc.pl.embedding(
    adata,
    basis="Concord_UMAP",
    color=["Sample", "Location", "Age", "BMI"],
    size=1,
    ncols=2,
    frameon=False,
    legend_loc='center left',
    show=False,
    sort_order=True
)

# =========================
# FORMAT ALL AXES
# =========================
fig = plt.gcf()
axes = fig.axes

for ax in axes:
    ax.set_title("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

    # rasterize points for PDF size
    for col in ax.collections:
        col.set_rasterized(True)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/UMAP_no93217162samples"

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", format="svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
marker_genes_dict = {
    "B-cell": ["CD79A", "CD19"],
    "Macrophages": ["CCR3", "CXCR3", "C1QC"],
    "Cadherins": ["CDH1", "CDH5"],
    "Beta Cells": ["IAPP", "GCK", "PDX1", "MAFA", 'UCN3', 'GLP1R', 'MAFB', 'NEUROD1', 'NKX6-1', 'PAX4', 'SOX1', 'SDC4'],
    "Pericytes": ["CSPG4", "PDGFRB", "SYP", 'ACE2', 'RGS5'],
    "Endothelial": ['PLVAP', 'PECAM1', 'VWF'],
    "Ductal Cells": ["CFTR", "KRT19", "DCDC2", 'PROM1', 'SOX9', 'SPP1', 'TUBB3', 'VAT1L'], 
    "Endocrine Markers": ['CHGA', 'UCHL1', 'ISL1', 'CCDC186', 'SIMC1', 'NOS1', 'ERLIN1'], 
    "Alpha Cells": ['GCG', 'BEND4', 'IRX1', 'IRX2', 'LOXL4', 'RPL14'], 
    "Delta Cells": ['SST', 'SSTR2', 'HHEX', 'BMI1', 'VIP', 'RSPH1'], 
    "Epsilon Cells": ['GHR', 'LGALS3BP', 'POGK', 'SERGEF', 'VTN'], 
    "PPY Cells": ['PPY', 'PAX6', 'SERTM1'], 
    "Nerve Cells": ['CBX2', 'CEACAM1', 'MNX1', 'NCAM1', 'NPY']
}

# SECTION 3 — LEIDEN CLUSTERING + CELL TYPE ANNOTATION
 res=1.0 yields 21 clusters; cluster 20 removed (low quality
 by QC metrics and no clear marker gene identity)
 Two annotation schemes created:
   merged_cell_type   — endocrine subtypes pooled for broad analysis
   specific_cell_type — retains edge alpha and plasma/alpha populations
 Annotation confirmed by: dotplot, Wilcoxon DEG, marker gene UMAP

In [ ]:
sc.pp.neighbors(adata, use_rep='Concord')
sc.tl.leiden(adata, resolution=1.0, key_added='concord_leiden_1')

In [ ]:
sc.pl.embedding(
    adata,
    basis='Concord_UMAP',
    color=['concord_leiden_1', 'Sample', 'transcript_count', 'Location' ],
    frameon=False,
    ncols=2,
    vmax=500
)

In [ ]:
adata.obs['concord_leiden_1'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(adata, "concord_leiden_1", use_raw=False, method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=20)

In [ ]:
sc.tl.rank_genes_groups(adata, "concord_leiden_1", use_raw=False, layer='counts', method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=20)

In [ ]:
sc.pl.dotplot(adata, marker_genes_dict, groupby='concord_leiden_1', vmax=10)

# SECTION 4 — MARKER GENE VISUALIZATION
 Dotplots and UMAP expression panels for all major cell types
 Exported as SI figures; gene panels grouped by cell compartment:
   endocrine (GCG, SST, PPY, CHGA)
   exocrine  (ALB, IGFBP2, CFTR, AQP1)
   immune    (CD8A, C1QC, CD4, C1QB)
   vascular  (PLVAP, PECAM1, PDGFRB, RGS5, COL1A1, COL6A3)

In [ ]:


def merge_clusters(adata, cluster_column, merge_dict, new_column_name="merged_clusters"):
    # Create a new column with original cluster labels
    adata.obs[new_column_name] = adata.obs[cluster_column].astype(str)

    # Apply merging rules
    for old_clusters, new_label in merge_dict.items():
        adata.obs.loc[adata.obs[cluster_column].isin(old_clusters), new_column_name] = new_label

    return adata


In [ ]:
import scanpy as sc

def rename_cell_categories(adata, column, rename_dict):
    """
    Rename categories in a specified column of an AnnData object.

    Parameters:
    - adata: AnnData object.
    - column: str, the column in adata.obs that contains categorical labels.
    - rename_dict: dict, mapping of old category names to new names.

    Returns:
    - Updated AnnData object with renamed categories.
    """
    if column in adata.obs:
        adata.obs[column] = adata.obs[column].astype("category")  # Ensure categorical dtype
        adata.obs[column] = adata.obs[column].cat.rename_categories(rename_dict)
    else:
        print(f"Column '{column}' not found in adata.obs.")
    
    return adata


In [ ]:
adataf = adata[
    ~adata.obs["concord_leiden_1"].isin(["20"])
].copy()

In [ ]:
sc.pl.dotplot(adataf, marker_genes_dict, groupby='concord_leiden_1', vmax=10)

In [ ]:
sc.tl.rank_genes_groups(adataf, "concord_leiden_1", use_raw=False, method="wilcoxon")
sc.pl.rank_genes_groups(adataf, n_genes=20)

In [ ]:
# Define clusters to merge
merge_map = {
    ('3', '7', '10', '9', '19'): 'Alpha Cells',
    ('1', '11'): 'Beta Cells',
    ('16', '17', '2'): 'Ductal Cells',
    ('0', '4', '6'): 'Acinar Cells',
    ('8', '13'): 'Endothelial Cells',
    ('5', '14'): 'Mesenchymal Cells',


    
}

# Merge clusters
adataf = merge_clusters(
    adataf,
    cluster_column="concord_leiden_1",
    merge_dict=merge_map,
    new_column_name="merged_cell_type"
)

# Check result
print(adataf.obs["merged_cell_type"].value_counts())

In [ ]:
sc.tl.rank_genes_groups(adataf, "merged_cell_type", use_raw=False, method="wilcoxon")
sc.pl.rank_genes_groups(adataf, n_genes=20)

In [ ]:


# Example usage:
rename_dict = {
    "15": "Immune Cells",

    "12": "Delta Cells",
    "18": "PPY Cells",
}

adataf = rename_cell_categories(adataf, column="merged_cell_type", rename_dict=rename_dict)

# Check renamed categories
print(adataf.obs["merged_cell_type"].cat.categories)

In [ ]:
# Define clusters to merge
merge_map = {
    ('3', '7', '10'): 'Alpha Cells',
    ('1', '11'): 'Beta Cells',
    ('16', '17', '2'): 'Ductal Cells',
    ('0', '4', '6'): 'Acinar Cells',
    ('8', '13'): 'Endothelial Cells',
    ('5', '14'): 'Mesenchymal Cells',


    
}

# Merge clusters
adataf = merge_clusters(
    adataf,
    cluster_column="concord_leiden_1",
    merge_dict=merge_map,
    new_column_name="specific_cell_type"
)

# Check result
print(adataf.obs["specific_cell_type"].value_counts())

In [ ]:
sc.tl.rank_genes_groups(adataf, "specific_cell_type", use_raw=False, method="wilcoxon")
sc.pl.rank_genes_groups(adataf, n_genes=20)

In [ ]:


# Example usage:
rename_dict = {
    "9": "Edge Alpha Cells",
    "15": "Immune Cells",
    "19": "Plasma/Alpha Cells", 
    "12": "Delta Cells",
    "18": "PPY Cells",
}

adataf = rename_cell_categories(adataf, column="specific_cell_type", rename_dict=rename_dict)

# Check renamed categories
print(adataf.obs["specific_cell_type"].cat.categories)

In [ ]:
sc.pl.embedding(
    adataf,
    basis='Concord_UMAP',
    color=['specific_cell_type', 'Sample', 'Location', 'merged_cell_type' ],
    frameon=False,
    ncols=2,
)

In [ ]:
ECM = ['SFN', 'COL4A3', 'COL18A1', 'COL15A1', 'COL5A3', 'COL6A3', 'COL6A2', 'COL5A2', 'COL1A1', 'COL1A2', 'COL6A1', 'COL3A1', 
       'LGALS4', 'LOXL4', 'SFRP2', 'SPARCL1', 'IGFBP7', 'AGPS', 'AGRN', 'CLASRP', 'MGP', 'AMBP', 'HSPG2', 
       'F13A1', 'ASPN', 'LAMC3', 'TINAGL1', 'VWA1', 'PCOLCE', 'MFGE8', 'FBLN2', 
       'LAMB2', 'FBN1', 'LAMA2', 'THBS1', 'VTN', 'FN1', 'FGG', 'FGB', 'FGA', 'LAMA5', 'COL16A1', 'COL11A1', 
       'COL4A2', 'COL4A1', 'COL2A1', 'OGN', 'EMILIN1', 'LAMA4', 'POSTN', 'FBN2', 'COL12A1', 'COL14A1', 'COL5A1', 'LAMC1', 'LAMC2',  'LAMB3', 'FBLN1',
        'LAMB1', 'LAMB4', 'LAMA3', 'SDC1']

In [ ]:
vasc_marker_genes_dict = {
    "Activated Stellate": ["PDGFRB", "COL1A1", "COL6A3", "DCN", "LAMA2", "LUM", "FN1", "MMP2"],
    "Quiescent Stellate": ["PDGFRB", "CSPG4", "COL4A1", "RGS5", "ACE2"],
    "Endothelial Cells": ["FLT1", "PLVAP", "VWF"],
    "Pericytes": ["CSPG4", "PDGFRB", "S100A1", "SYP", 'C1R', 'CALD1', 'RGS5'],
    "Immuno Associated Vasculature": ['ICAM1', 'C1R', 'AIF1', 'IL4', 'CXCR3', 'C1S'],
    "Endocrine Markers": ['CHGA', 'GCG', 'SST', 'PPY'], 
    "Nerve Cells": ['CBX2', 'CEACAM1', 'GRIA2', 'MNX1', 'NCAM1', 'NPY', 'TRPV1']
}

In [ ]:
ECM_dict = { 
    "Collagens": ['PCOLCE', 'COL1A1', 'COL14A1', 'COL5A3', 'COL1A2', 'COL6A2', 'COL3A1', 'COL18A1', 'COL4A2', 'COL11A1', 
       'COL5A1', 'COL16A1', 'COL5A2', 'COL15A1', 'COL4A1', 'COL4A3', 'COL12A1', 'COL6A3', 'COL2A1', 'COL6A1'],
    "Laminins": ['LAMA4', 'LAMA5', 'LAMC1', 'LAMC2', 'LAMB2', 'LAMB3', 'LAMB1', 'LAMA2', 'LAMC3', 'LAMB4', 'LAMA3'],
    "Others": ['FBLN1','FGB','FGA','FGG','FBLN2','VWA1','HSPG2','FBN2','SDC1','F13A1','AMBP','MFGE8','TINAGL1','SFRP2',
               'SPARCL1','EMILIN1','FN1','THBS1','LOXL4','CLASRP','OGN','VTN','MGP','AGPS','LGALS4','SFN','ASPN','AGRN','POSTN']
}

# SECTION 5 — DEG EXPORT (SI TABLES)
 Wilcoxon rank-sum DEG for both annotation schemes
 Thresholds: FDR < 0.05, |logFC| > 0.25
 Full and filtered tables exported as CSV for SI

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

# Run DEG
sc.tl.rank_genes_groups(
    adataf,
    "merged_cell_type",
    use_raw=False,
    method="wilcoxon"
)

# Plot (don't show yet)
sc.pl.rank_genes_groups(
    adataf,
    n_genes=20,
    show=False
)

# Save
plt.savefig(
    "/Users/kmeneses/Desktop/Figure2_Data/rank_genes_groups.svg",
    format="svg",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/Figure2_Data/rank_genes_groups.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

# Run DEG
sc.tl.rank_genes_groups(
    adataf,
    "specific_cell_type",
    use_raw=False,
    method="wilcoxon"
)

# Plot (don't show yet)
sc.pl.rank_genes_groups(
    adataf,
    n_genes=20,
    show=False
)

# Save
plt.savefig(
    "/Users/kmeneses/Desktop/Figure2_Data/rank_genes_groups_s.svg",
    format="svg",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/Figure2_Data/rank_genes_groups_s.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import matplotlib as mpl

# Keep text editable in Illustrator
mpl.rcParams["svg.fonttype"] = "none"

# -------------------------
# PLOT (don't show yet)
# -------------------------
sc.pl.dotplot(
    adataf,
    marker_genes_dict,
    groupby='merged_cell_type',
    vmax=5,
    show=False
)

# Get figure
fig = plt.gcf()

# Optional: set size for consistency
fig.set_size_inches(17, 4)

# -------------------------
# SAVE
# -------------------------
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/dotplot_cellsmerged"

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", format="svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import matplotlib as mpl

# Keep text editable in Illustrator
mpl.rcParams["svg.fonttype"] = "none"

# -------------------------
# PLOT (don't show yet)
# -------------------------
sc.pl.dotplot(
    adataf,
    marker_genes_dict,
    groupby='specific_cell_type',
    vmax=5,
    show=False
)

# Get figure
fig = plt.gcf()

# Optional: set size for consistency
fig.set_size_inches(17, 4)

# -------------------------
# SAVE
# -------------------------
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/dotplot_cellspecific"

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", format="svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import matplotlib as mpl

mpl.rcParams["svg.fonttype"] = "none"

# -------------------------
# PLOT (let scanpy create fig)
# -------------------------
fig = sc.pl.embedding(
    adataf,
    basis='Concord_UMAP',
    color=["GCG", "SST", "PPY", "CHGA"],
    frameon=False,
    ncols=4,
    size=0.1,
    vmax=5,
    use_raw=False,
    colorbar_loc=None,
    show=False,
    return_fig=True   # 🔥 CRITICAL
)

axes = fig.axes

# -------------------------
# Rasterize points
# -------------------------
for ax in axes:
    for col in ax.collections:
        col.set_rasterized(True)

# -------------------------
# ADD ONE SHARED COLORBAR
# -------------------------
# find first axis that actually has data
mappable = None
for ax in axes:
    if len(ax.collections) > 0:
        mappable = ax.collections[0]
        break

if mappable is not None:
    cbar = fig.colorbar(
        mappable,
        ax=axes,
        fraction=0.02,
        pad=0.02
    )
    cbar.set_label("Expression", fontsize=8)

# -------------------------
# SET FIG SIZE (🔥 move here)
# -------------------------
fig.set_size_inches(3,1)

plt.tight_layout()

# -------------------------
# SAVE
# -------------------------
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/umap_expression_small_edogenes"

fig.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import matplotlib as mpl

mpl.rcParams["svg.fonttype"] = "none"

# -------------------------
# PLOT (let scanpy create fig)
# -------------------------
fig = sc.pl.embedding(
    adataf,
    basis='Concord_UMAP',
    color=["ALB", "IGFBP2", "CFTR", "AQP1"],
    frameon=False,
    ncols=4,
    size=0.1,
    vmax=4,
    use_raw=False,
    colorbar_loc=None,
    show=False,
    return_fig=True   # 🔥 CRITICAL
)

axes = fig.axes

# -------------------------
# Rasterize points
# -------------------------
for ax in axes:
    for col in ax.collections:
        col.set_rasterized(True)

# -------------------------
# ADD ONE SHARED COLORBAR
# -------------------------
# find first axis that actually has data
mappable = None
for ax in axes:
    if len(ax.collections) > 0:
        mappable = ax.collections[0]
        break

if mappable is not None:
    cbar = fig.colorbar(
        mappable,
        ax=axes,
        fraction=0.02,
        pad=0.02
    )
    cbar.set_label("Expression", fontsize=8)

# -------------------------
# SET FIG SIZE (🔥 move here)
# -------------------------
fig.set_size_inches(3,1)

plt.tight_layout()

# -------------------------
# SAVE
# -------------------------
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/umap_expression_small_exocrinegenes"

fig.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import matplotlib as mpl

mpl.rcParams["svg.fonttype"] = "none"

# -------------------------
# PLOT (let scanpy create fig)
# -------------------------
fig = sc.pl.embedding(
    adataf,
    basis='Concord_UMAP',
    color=["CD8A", "C1QC", "CD4", "C1QB"],
    frameon=False,
    ncols=4,
    size=0.1,
    vmax=2,
    colorbar_loc=None,
    show=False,
    use_raw=False,
    return_fig=True   # 🔥 CRITICAL
)

axes = fig.axes

# -------------------------
# Rasterize points
# -------------------------
for ax in axes:
    for col in ax.collections:
        col.set_rasterized(True)

# -------------------------
# ADD ONE SHARED COLORBAR
# -------------------------
# find first axis that actually has data
mappable = None
for ax in axes:
    if len(ax.collections) > 0:
        mappable = ax.collections[0]
        break

if mappable is not None:
    cbar = fig.colorbar(
        mappable,
        ax=axes,
        fraction=0.02,
        pad=0.02
    )
    cbar.set_label("Expression", fontsize=8)

# -------------------------
# SET FIG SIZE (🔥 move here)
# -------------------------
fig.set_size_inches(3,1)

plt.tight_layout()

# -------------------------
# SAVE
# -------------------------
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/umap_expression_small_immunegenes"

fig.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import matplotlib as mpl

mpl.rcParams["svg.fonttype"] = "none"

# -------------------------
# PLOT (let scanpy create fig)
# -------------------------
fig = sc.pl.embedding(
    adataf,
    basis='Concord_UMAP',
    color=["PLVAP", "PECAM1", "PDGFRB", "RGS5", "COL1A1", "COL6A3", "DCN", "COL6A2" ],
    frameon=False,
    ncols=4,
    size=0.1,
    vmax=3,
    use_raw=False,
    colorbar_loc=None,
    show=False,
    return_fig=True   # 🔥 CRITICAL
)

axes = fig.axes

# -------------------------
# Rasterize points
# -------------------------
for ax in axes:
    for col in ax.collections:
        col.set_rasterized(True)

# -------------------------
# ADD ONE SHARED COLORBAR
# -------------------------
# find first axis that actually has data
mappable = None
for ax in axes:
    if len(ax.collections) > 0:
        mappable = ax.collections[0]
        break

if mappable is not None:
    cbar = fig.colorbar(
        mappable,
        ax=axes,
        fraction=0.02,
        pad=0.02
    )
    cbar.set_label("Expression", fontsize=8)

# -------------------------
# SET FIG SIZE (🔥 move here)
# -------------------------
fig.set_size_inches(3,2)

plt.tight_layout()

# -------------------------
# SAVE
# -------------------------
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/umap_expression_small_vasculargenes"

fig.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import scanpy as sc
import pandas as pd

# =========================
# 1. RUN DEG (all cell types)
# =========================
sc.tl.rank_genes_groups(
    adataf,
    groupby="merged_cell_type",   # or "cell_type_p"
    method="wilcoxon",
    use_raw=False
)

# =========================
# 2. EXTRACT ALL RESULTS
# =========================
groups = adataf.obs["merged_cell_type"].cat.categories

deg_list = []

for g in groups:
    df = sc.get.rank_genes_groups_df(adataf, group=g)
    df["group"] = g
    deg_list.append(df)

deg_all = pd.concat(deg_list)

# =========================
# 3. CLEAN + FORMAT
# =========================
deg_all = deg_all.rename(columns={
    "names": "gene",
    "scores": "score",
    "logfoldchanges": "logFC",
    "pvals": "pval",
    "pvals_adj": "pval_adj"
})

deg_all = deg_all.sort_values(["group", "pval_adj"])

# =========================
# 4. EXPORT FULL TABLE (SI)
# =========================
out_full = "/Users/kmeneses/Desktop/Figure_2_plots/SI/DEG_all_celltypes.csv"
deg_all.to_csv(out_full, index=False)

# =========================
# 5. FILTER SIGNIFICANT GENES (OPTIONAL)
# =========================
deg_sig = deg_all[
    (deg_all["pval_adj"] < 0.05) &
    (deg_all["logFC"].abs() > 0.25)
]

out_sig = "/Users/kmeneses/Desktop/Figure_2_plots/SI/DEG_significant.csv"
deg_sig.to_csv(out_sig, index=False)

# =========================
# DONE
# =========================
print("Saved:")
print(out_full)
print(out_sig)

In [ ]:
import scanpy as sc
import pandas as pd

# =========================
# 1. RUN DEG (all cell types)
# =========================
sc.tl.rank_genes_groups(
    adataf,
    groupby="specific_cell_type",   # or "cell_type_p"
    method="wilcoxon",
    use_raw=False
)

# =========================
# 2. EXTRACT ALL RESULTS
# =========================
groups = adataf.obs["specific_cell_type"].cat.categories

deg_list = []

for g in groups:
    df = sc.get.rank_genes_groups_df(adataf, group=g)
    df["group"] = g
    deg_list.append(df)

deg_all = pd.concat(deg_list)

# =========================
# 3. CLEAN + FORMAT
# =========================
deg_all = deg_all.rename(columns={
    "names": "gene",
    "scores": "score",
    "logfoldchanges": "logFC",
    "pvals": "pval",
    "pvals_adj": "pval_adj"
})

deg_all = deg_all.sort_values(["group", "pval_adj"])

# =========================
# 4. EXPORT FULL TABLE (SI)
# =========================
out_full = "/Users/kmeneses/Desktop/Figure_2_plots/SI/DEG_all_celltypes_specific.csv"
deg_all.to_csv(out_full, index=False)

# =========================
# 5. FILTER SIGNIFICANT GENES (OPTIONAL)
# =========================
deg_sig = deg_all[
    (deg_all["pval_adj"] < 0.05) &
    (deg_all["logFC"].abs() > 0.25)
]

out_sig = "/Users/kmeneses/Desktop/Figure_2_plots/SI/DEG_significant_specific.csv"
deg_sig.to_csv(out_sig, index=False)

# =========================
# DONE
# =========================
print("Saved:")
print(out_full)
print(out_sig)

In [ ]:
adataf.write_h5ad('/Users/kmeneses/Desktop/updated_data/healthycells_allsamples_no9317162_labeledCV.h5ad')

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

# =========================
# PLOT
# =========================


palette = {
    "Acinar Cells": "#e67bda",
    "Ductal Cells": "#c0c9c0",
    "Mesenchymal Cells": "#2416f0",
    "Endothelial Cells": "#33a02c",
    "Immune Cells": "#eff21c",
    "Beta Cells": "#e70f0f",
    "Alpha Cells": "#dd9014",
    "Delta Cells": "#8f3cdc",
    "PPY Cells": "#12ccec",

}


sc.pl.embedding(
    adataf,
    basis="Concord_UMAP",
    color=["merged_cell_type"],
    size=2,
    palette=palette,
    ncols=2,
    frameon=False,
    legend_loc='center left',
    show=False,
    sort_order=True
)

# =========================
# FORMAT ALL AXES
# =========================
fig = plt.gcf()
axes = fig.axes

for ax in axes:
    ax.set_title("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

    # rasterize points for PDF size
    for col in ax.collections:
        col.set_rasterized(True)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/UMAP_no93217162samples_filtered_leidneclusters"

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", format="svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import numpy as np

# =========================
# DEFINE GROUPS
# =========================
keep_cells = [
    "Mesenchymal Cells",
    "Endothelial Cells",
]

# masks
is_vasc = adataf.obs["merged_cell_type"].isin(keep_cells)

adata_bg = adataf[~is_vasc]   # background
adata_fg = adataf[is_vasc]    # foreground

# =========================
# PALETTE
# =========================
palette = {
    "Mesenchymal Cells": "#2416f0",
    "Endothelial Cells": "#33a02c",
}

# =========================
# PLOT
# =========================
fig, ax = plt.subplots(figsize=(4,4))

# ---- BACKGROUND (grey, transparent)
sc.pl.embedding(
    adata_bg,
    basis="Concord_UMAP",
    color=None,
    size=2,
    frameon=False,
    show=False,
    ax=ax
)

# manually set grey + transparency
for col in ax.collections:
    col.set_color("#d3d3d3")
    col.set_alpha(0.3)
    col.set_rasterized(True)

# ---- FOREGROUND (colored vascular)
sc.pl.embedding(
    adata_fg,
    basis="Concord_UMAP",
    color="merged_cell_type",
    size=3,
    palette=palette,
    frameon=False,
    legend_loc='center left',
    show=False,
    ax=ax
)

# rasterize foreground too
for col in ax.collections:
    col.set_rasterized(True)

# =========================
# CLEAN FORMAT
# =========================
ax.set_title("")
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")

for spine in ax.spines.values():
    spine.set_linewidth(1)

plt.tight_layout()

# =========================
# SAVE
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/UMAP_vascular_context_broad"

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
# clean labels (important)
adataf.obs["Location"] = (
    adataf.obs["Location"]
    .astype(str)
    .str.strip()
)

# assign colors in correct order
categories = adataf.obs["Location"].astype("category").cat.categories

color_map = {
    "Islet": "#008080",     # yellow
    "Exocrine": "#FFD700"   # teal
}

adataf.uns["Location_colors"] = [
    color_map[c] for c in categories
]

# -------------------------
# PLOT
# -------------------------
sc.pl.embedding(
    adataf,
    basis="Concord_UMAP",
    color=["Sample", "Location"],
    size=2,
    ncols=2,
    frameon=False,
    legend_loc='center left',
    show=False,
    sort_order=True
)


# =========================
# FORMAT ALL AXES
# =========================
fig = plt.gcf()
axes = fig.axes

for ax in axes:
    ax.set_title("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

    for col in ax.collections:
        col.set_rasterized(True)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/UMAP_no93217162samples_filtered_locationsamples"

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", format="svg", dpi=600, bbox_inches="tight")

plt.show()

# SECTION 6 — ISLET ROI GEOMETRY QC
 Parse per-sample ROI CSV files (Shapely WKT format)
 Compute: area (µm²), perimeter (µm), circularity per islet
 Outliers flagged at 1st/99th percentile of area
 circularity = 4π × area / perimeter² (1.0 = perfect circle)
 Results stored in adataf.uns["islet_geometry"] + exported CSV

In [ ]:
import pandas as pd
import numpy as np
import glob
from shapely import from_wkt
# Load per-sample cell boundary parquet files (WKB/WKT geometry)
# Build cell_polygons dict {EntityID: shapely polygon}
# Build cell_centroids dict {EntityID: (x, y)}
# Used for all downstream spatial distance calculations
# -------------------------------------
# FOLDER WITH ALL ROI CSV FILES
# -------------------------------------

roi_folder = "/Volumes/Samsung_4TB/Islets2_ROI_CSV_Merscope/"

roi_files = glob.glob(f"{roi_folder}/*.csv")

print("Total ROI files:", len(roi_files))


# -------------------------------------
# STORAGE
# -------------------------------------

islet_results = []


# -------------------------------------
# LOOP THROUGH ROI FILES
# -------------------------------------

for f in roi_files:

    df = pd.read_csv(f)

    sample = df["dataset"].iloc[0]

    print("Processing:", sample)

    # detect ID column
    if "ROI" in df.columns:
        id_col = "ROI"
    elif "EntityID" in df.columns:
        id_col = "EntityID"
    else:
        raise ValueError(f"No ROI or EntityID column in {f}")

    # safe geometry parsing
    for i, row in df.iterrows():

        try:
            poly = from_wkt(row["geometry"])

            if poly is None or poly.is_empty:
                continue

            area = poly.area
            perimeter = poly.length
            centroid = poly.centroid

            islet_results.append({

                "Sample": sample,
                "ROI_ID": row[id_col],

                "islet_area_um2": area,
                "islet_perimeter_um": perimeter,

                "centroid_x": centroid.x,
                "centroid_y": centroid.y
            })

        except Exception as e:
            print(f"Skipping invalid ROI in {sample} row {i}")


# -------------------------------------
# COMBINE RESULTS
# -------------------------------------

islet_df = pd.DataFrame(islet_results)

print(islet_df.head())


# -------------------------------------
# SAVE OUTPUT
# -------------------------------------

out_path = "/Users/kmeneses/Desktop/islet_geometry_metrics.csv"

islet_df.to_csv(out_path, index=False)

print("Saved:", out_path)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import os
# For each cell: find nearest islet ROI using STRtree
# Compute true edge-to-edge distance (negative = inside islet)
# Pixel → µm conversion: × 0.108
# Stored as adataf.obs["dist_edge_islet_um"]

# -------------------------------------
# STYLE
# -------------------------------------
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# -------------------------------------
# OUTPUT DIR
# -------------------------------------
outdir = (
    "/Users/kmeneses/Desktop/"
    "Figure_2_plots/SI/islet_geometry_QC"
)

os.makedirs(outdir, exist_ok=True)

# -------------------------------------
# LOAD ROI TABLE
# -------------------------------------
df = pd.read_csv(
    "/Users/kmeneses/Desktop/islet_geometry_metrics.csv"
)

print("Total islets:", len(df))

print("\nIslets per sample")
print(df["Sample"].value_counts())

# -------------------------------------
# BASIC STATS
# -------------------------------------
print("\nArea statistics")
print(df["islet_area_um2"].describe())

print("\nPerimeter statistics")
print(df["islet_perimeter_um"].describe())

# -------------------------------------
# HISTOGRAM:
# AREA
# -------------------------------------
plt.figure(figsize=(6,4))

sns.histplot(
    df["islet_area_um2"],
    bins=50
)

plt.xlabel("Islet area (µm²)")
plt.ylabel("Count")
plt.title("Islet area distribution")

sns.despine()

plt.tight_layout()

plt.savefig(
    f"{outdir}/islet_area_distribution.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{outdir}/islet_area_distribution.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{outdir}/islet_area_distribution.pdf",
    bbox_inches="tight"
)

plt.show()

# -------------------------------------
# HISTOGRAM:
# PERIMETER
# -------------------------------------
plt.figure(figsize=(6,4))

sns.histplot(
    df["islet_perimeter_um"],
    bins=50
)

plt.xlabel("Islet perimeter (µm)")
plt.ylabel("Count")
plt.title("Islet perimeter distribution")

sns.despine()

plt.tight_layout()

plt.savefig(
    f"{outdir}/islet_perimeter_distribution.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{outdir}/islet_perimeter_distribution.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{outdir}/islet_perimeter_distribution.pdf",
    bbox_inches="tight"
)

plt.show()

# -------------------------------------
# SCATTER:
# AREA VS PERIMETER
# -------------------------------------
plt.figure(figsize=(5,5))

sns.scatterplot(
    data=df,
    x="islet_area_um2",
    y="islet_perimeter_um",
    alpha=0.6,
    s=20
)

plt.xlabel("Islet area (µm²)")
plt.ylabel("Islet perimeter (µm)")
plt.title("Islet geometry relationship")

sns.despine()

plt.tight_layout()

plt.savefig(
    f"{outdir}/islet_area_vs_perimeter.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{outdir}/islet_area_vs_perimeter.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{outdir}/islet_area_vs_perimeter.pdf",
    bbox_inches="tight"
)

plt.show()

# -------------------------------------
# DETECT OUTLIERS
# -------------------------------------
area_q1 = df["islet_area_um2"].quantile(0.01)
area_q99 = df["islet_area_um2"].quantile(0.99)

outliers = df[
    (df["islet_area_um2"] < area_q1) |
    (df["islet_area_um2"] > area_q99)
]

print("\nPotential area outliers:")
print(
    outliers[
        ["Sample","ROI_ID","islet_area_um2"]
    ]
)

# save outliers
outliers.to_csv(
    f"{outdir}/islet_area_outliers.csv",
    index=False
)

# -------------------------------------
# CIRCULARITY QC
# -------------------------------------
df["circularity"] = (
    4 * np.pi * df["islet_area_um2"] /
    (df["islet_perimeter_um"]**2)
)

plt.figure(figsize=(6,4))

sns.histplot(
    df["circularity"],
    bins=50
)

plt.xlabel("Circularity")
plt.ylabel("Count")
plt.title("Islet shape QC")

sns.despine()

plt.tight_layout()

plt.savefig(
    f"{outdir}/islet_circularity_distribution.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{outdir}/islet_circularity_distribution.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{outdir}/islet_circularity_distribution.pdf",
    bbox_inches="tight"
)

plt.show()

print("\nCircularity stats")
print(df["circularity"].describe())

print("\n✔ DONE — plots exported")

In [ ]:
adataf.obs_names

In [ ]:
adataf.obs_names = adataf.obs_names.str.split("_").str[0]

In [ ]:
adataf.obs_names

In [ ]:
coords = adataf.obsm["spatial"]

In [ ]:
islet_df["circularity"] = (
    4 * np.pi * islet_df["islet_area_um2"] /
    (islet_df["islet_perimeter_um"] ** 2)
)

In [ ]:
adataf.uns["islet_geometry"] = islet_df

In [ ]:
adataf.uns["islet_geometry"]

# SECTION 7 — LOAD CELL BOUNDARY POLYGONS
 Read MERFISH cell_boundaries.parquet per sample
 Handles both WKB (binary) and WKT (text) geometry formats
 Deduplicates cells across tiles using EntityID
 Outputs: cell_polygons {EntityID: polygon}
          cell_centroids {EntityID: (x, y)}
 These dicts are required for all downstream distance calculations

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
from shapely import from_wkb, from_wkt

# -----------------------------
# DEFINE PATHS FIRST (OUTSIDE LOOP)
# -----------------------------
sample_to_parquet = {
    "HuP03A": [
        "/Volumes/Samsung_4TB/HuP03A_L_data/cell_boundaries.parquet",
        "/Volumes/Samsung_4TB/HuP03A_S_data/cell_boundaries.parquet"
    ],
    "HuP20A": "/Volumes/Samsung_4TB/HuP20A_data/cell_boundaries.parquet",
    "HuP31A": "/Volumes/Samsung_4TB/HuP_31A_data/cell_boundaries.parquet",
    "HuP53A": "/Volumes/Samsung_4TB/HuP53A_data_new/cell_boundaries.parquet",
    "HuP67A": "/Volumes/Samsung_4TB/HuP67A_data/cell_boundaries.parquet",
    "HuP71A": "/Volumes/Samsung_4TB/HuP71A_data/cell_boundaries.parquet",
    "RHIP156": "/Volumes/Samsung_4TB/HuP156_data/cell_boundaries.parquet"
}

# -----------------------------
# INIT STORAGE
# -----------------------------
cell_polygons = {}
cell_centroids = {}

print("Loading cell boundaries...")

# -----------------------------
# LOOP THROUGH FILES
# -----------------------------
for sample, paths in sample_to_parquet.items():

    if not isinstance(paths, list):
        paths = [paths]

    for path in paths:

        print("Reading:", path)

        table = pq.read_table(path).to_pandas()

        # find geometry column
        geom_candidates = [c for c in table.columns if "geom" in c.lower()]
        if len(geom_candidates) == 0:
            raise ValueError(f"No geometry column found in {path}")

        geom_col = geom_candidates[0]

        ids = table["EntityID"].astype(str)
        g = table[geom_col].dropna()

        if len(g) == 0:
            continue

        first_val = g.iloc[0]

        if isinstance(first_val, (bytes, bytearray, memoryview)):
            polys = from_wkb(g.values)
        else:
            polys = from_wkt(g.values)

        for eid, poly in zip(ids, polys):

            key = str(eid)

            # skip duplicates across tiles
            if key in cell_polygons:
                continue

            cell_polygons[key] = poly

            c = poly.centroid
            cell_centroids[key] = (c.x, c.y)

print("Loaded unique polygons:", len(cell_polygons))

In [ ]:
mpl.rcParams["svg.fonttype"] = "none"


# SECTION 8 — SPATIAL DISTANCE: CELL → ISLET BOUNDARY
 For each cell: find nearest islet ROI polygon using STRtree
 Signed distance: negative = inside islet, positive = outside
 Pixel → µm: no conversion needed (ROI coordinates already in µm)
 NaN = cell polygon not found in boundary files
 Stored as adataf.obs["dist_edge_islet_um"]

In [ ]:
import glob
import os
import numpy as np
import pandas as pd

from shapely import from_wkt
from shapely.strtree import STRtree

# =========================================================
# SETTINGS
# =========================================================
ROI_FOLDER = "/Volumes/Samsung_4TB/Islets2_ROI_CSV_Merscope/"

# =========================================================
# FIND ROI FILES
# =========================================================
roi_files = glob.glob(
    f"{ROI_FOLDER}/*.csv"
)

print(f"Found {len(roi_files)} ROI files\n")

# =========================================================
# VALID SAMPLES
# =========================================================
valid_samples = set(
    adataf.obs["Sample"].astype(str).unique()
)

print("Valid samples:")
print(valid_samples)

# =========================================================
# LOAD ROI POLYGONS
# =========================================================
sample_roi_lists = {}

for f in roi_files:

    fname = os.path.basename(f)

    print(f"\nProcessing: {fname}")

    # -------------------------------------------------
    # SAMPLE ID
    # -------------------------------------------------
    sample_id = (
        fname
        .replace(".csv", "")
        .split("_")[0]
    )

    print(f"Parsed sample: {sample_id}")

    if sample_id not in valid_samples:

        print("⚠ Not in adata")
        continue

    df = pd.read_csv(f)

    # -------------------------------------------------
    # FIND GEOMETRY COLUMN
    # -------------------------------------------------
    geom_cols = [
        c for c in df.columns
        if "geom" in c.lower()
    ]

    if len(geom_cols) == 0:

        print("⚠ No geometry column")
        continue

    geom_col = geom_cols[0]

    rois = []

    for val in df[geom_col]:

        if pd.isna(val):
            continue

        try:

            poly = from_wkt(val)

            if (
                poly.is_empty or
                (not poly.is_valid)
            ):
                continue

            rois.append(poly)

        except:
            continue

    print(f"Loaded {len(rois)} ROIs")

    if len(rois) > 0:

        sample_roi_lists[sample_id] = rois

# =========================================================
# BUILD TREES
# =========================================================
sample_trees = {
    s: STRtree(rois)
    for s, rois in sample_roi_lists.items()
}

print("\n✔ Trees built:", len(sample_trees))

# =========================================================
# COMPUTE TRUE EDGE DISTANCES
# =========================================================
distances = np.full(
    adataf.n_obs,
    np.nan
)

for sample_id in adataf.obs["Sample"].unique():

    if sample_id not in sample_trees:
        continue

    tree = sample_trees[sample_id]
    rois = sample_roi_lists[sample_id]

    idxs = np.where(
        adataf.obs["Sample"] == sample_id
    )[0]

    print(
        f"Processing {len(idxs)} cells "
        f"for {sample_id}"
    )

    for idx in idxs:

        cid = str(adataf.obs_names[idx])

        # -------------------------------------------------
        # GET CELL POLYGON
        # -------------------------------------------------
        poly = cell_polygons.get(cid)

        if (
            poly is None or
            poly.is_empty or
            (not poly.is_valid)
        ):
            continue

        centroid = poly.centroid

        # -------------------------------------------------
        # FIND NEAREST ROI
        # -------------------------------------------------
        nearest = tree.nearest(centroid)

        # shapely 2 compatibility
        if hasattr(nearest, "geom_type"):

            roi = nearest

        else:

            roi = rois[nearest]

        # -------------------------------------------------
        # TRUE EDGE-TO-EDGE DISTANCE
        # -------------------------------------------------
        dist = poly.distance(roi.boundary)

        # -------------------------------------------------
        # NEGATIVE IF INSIDE ROI
        # -------------------------------------------------
        if roi.contains(centroid):

            dist = -dist

        distances[idx] = dist

# =========================================================
# STORE
# =========================================================
adataf.obs["dist_edge_islet_um"] = distances

print("\n✔ DONE")

# =========================================================
# SANITY CHECK
# =========================================================
print(
    adataf.obs["dist_edge_islet_um"]
    .describe()
)

inside_pct = (
    adataf.obs["dist_edge_islet_um"] < 0
).mean() * 100

print(
    f"% inside islet: "
    f"{inside_pct:.2f}%"
)

In [ ]:
# fraction inside islets
inside_pct = (
    adataf.obs["dist_edge_islet_um"] < 0
).mean() * 100

print("Inside islet %:", inside_pct)

# median inside distance
print(
    adataf.obs.loc[
        adataf.obs["dist_edge_islet_um"] < 0,
        "dist_edge_islet_um"
    ].median()
)

# median outside distance
print(
    adataf.obs.loc[
        adataf.obs["dist_edge_islet_um"] > 0,
        "dist_edge_islet_um"
    ].median()
)

In [ ]:
# =====================================================
# CHECK DISTANCE COVERAGE PER SAMPLE
# =====================================================

DIST_COL = "dist_edge_islet_um"

summary = (
    adataf.obs
    .groupby("Sample")[DIST_COL]
    .agg(
        total_cells="size",
        non_nan=lambda x: x.notna().sum(),
        nan_count=lambda x: x.isna().sum()
    )
)

summary["percent_with_distance"] = (
    summary["non_nan"] /
    summary["total_cells"]
) * 100

print(summary)

# =====================================================
# SHOW ONLY PROBLEM DONORS
# =====================================================
print("\nDonors with missing distances:\n")

print(
    summary[
        summary["nan_count"] > 0
    ]
)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

sc.pl.violin(
    adataf,
    keys="dist_edge_islet_um",
    groupby="specific_cell_type",
    rotation=90,
    stripplot=True,
    show=False
)

plt.savefig(
    "/Users/kmeneses/Desktop/Figure_2_plots/SI/dist_to_islet_violin_by_specoficcelltype.svg",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SMALLER FIGURE
# =========================
plt.rcParams["figure.figsize"] = (4, 2)

# =========================
# VIOLIN
# =========================
sc.pl.violin(
    adataf,
    keys="dist_edge_islet_um",
    groupby="merged_cell_type",
    rotation=45,
    stripplot=False,   # 🔥 huge file-size reduction
    jitter=False,
    show=False
)

# =========================
# FORMAT
# =========================
fig = plt.gcf()

for ax in fig.axes:

    ax.set_ylabel("Distance to islet edge (µm)")

    ax.tick_params(
        axis="x",
        labelsize=6
    )

    ax.tick_params(
        axis="y",
        labelsize=6
    )

    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = (
    "/Users/kmeneses/Desktop/"
    "Figure_2_plots/SI/"
    "dist_to_islet_violin_by_mergedcelltype"
)

# PNG usually much smaller + cleaner
plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

# rasterized PDF smaller than SVG
plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

# optional SVG
plt.savefig(
    f"{out_base}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

d = adataf.obs["dist_edge_islet_um"].dropna()

print("=== DISTANCE SUMMARY (µm) ===")
print(f"Total cells: {len(d)}")
print(f"Inside islet (<0): {(d < 0).sum()} ({100*(d<0).mean():.1f}%)")
print(f"Outside islet (>0): {(d > 0).sum()} ({100*(d>0).mean():.1f}%)")

# =========================
# PLOT
# =========================
fig, axes = plt.subplots(
    1, 2,
    figsize=(10,3)
)

# full
axes[0].hist(
    d,
    bins=100,
    color="steelblue"
)

axes[0].axvline(
    0,
    color="red",
    linestyle="--",
    linewidth=1
)

axes[0].set_xlabel("Distance to islet edge (µm)")
axes[0].set_ylabel("Cell count")
axes[0].set_title("Full distribution")

# zoom
zoomed = d[d.between(-50, 50)]

axes[1].hist(
    zoomed,
    bins=100,
    color="steelblue"
)

axes[1].axvline(
    0,
    color="red",
    linestyle="--",
    linewidth=1
)

axes[1].set_xlabel("Distance to islet edge (µm)")
axes[1].set_title("Zoomed ±50 µm")

plt.tight_layout()

# =========================
# SAVE
# =========================
plt.savefig(
    "/Users/kmeneses/Desktop/Figure_2_plots/SI/islet_distance_distribution.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# SECTION 9 — SPATIAL DISTANCES: CELL → NEAREST CELL TYPE
 Three separate STRtree distance calculations per sample:
   dist_edge_endo_um        → nearest Endothelial Cell
   dist_edge_mesenchymal_um → nearest Mesenchymal Cell
   dist_edge_beta_um        → nearest Beta Cell
 All use edge-to-edge polygon distance (not centroid-to-centroid)
 Pixel → µm conversion: × 0.108 (MERFISH pixel size)
 Used downstream for spatial enrichment and proximity analysis

In [ ]:
import numpy as np
from shapely.strtree import STRtree

# =========================================================
# INIT
# =========================================================
dist_edge_endo = np.full(
    adataf.n_obs,
    np.nan
)

# =========================================================
# LOOP THROUGH SAMPLES
# =========================================================
for sample in adataf.obs["Sample"].unique():

    print("Processing:", sample)

    # -------------------------------------------------
    # SAMPLE MASK
    # -------------------------------------------------
    mask = (
        adataf.obs["Sample"] == sample
    )

    idx_in_adata = np.where(mask)[0]

    cells_in_sample = (
        adataf.obs_names[mask]
        .astype(str)
    )

    # -------------------------------------------------
    # PERICYTES IN SAMPLE
    # -------------------------------------------------
    peri_mask = (
        mask &
        (
            adataf.obs["merged_cell_type"]
            == "Endothelial Cells"
        )
    )

    peri_ids = (
        adataf.obs_names[peri_mask]
        .astype(str)
    )

    peri_polys = [
        cell_polygons[c]
        for c in peri_ids
        if c in cell_polygons
    ]

    if len(peri_polys) == 0:

        print(
            f"  No pericytes in {sample}. "
            f"Skipping."
        )

        continue

    # -------------------------------------------------
    # BUILD TREE
    # -------------------------------------------------
    tree = STRtree(peri_polys)

    peri_geoms_arr = np.array(
        peri_polys,
        dtype=object
    )

    # -------------------------------------------------
    # COMPUTE DISTANCES
    # -------------------------------------------------
    for i, cid in zip(
        idx_in_adata,
        cells_in_sample
    ):

        poly = cell_polygons.get(cid)

        if (
            poly is None or
            poly.is_empty
        ):
            continue

        nearest = tree.nearest(poly)

        if nearest is None:
            continue

        # shapely compatibility
        if hasattr(nearest, "geom_type"):

            nearest_geom = nearest

        else:

            nearest_geom = peri_geoms_arr[nearest]

        # -------------------------------------------------
        # TRUE EDGE-TO-EDGE DISTANCE
        # -------------------------------------------------
        d = poly.distance(nearest_geom)

        # convert pixels -> µm
        dist_edge_endo[i] = d * 0.108

# =========================================================
# STORE
# =========================================================
adataf.obs[
    "dist_edge_endo_um"
] = dist_edge_endo

print("\n✔ DONE")

# =========================================================
# SANITY CHECK
# =========================================================
print(
    adataf.obs[
        "dist_edge_endo_um"
    ].describe()
)

In [ ]:
import numpy as np
from shapely.strtree import STRtree

# =========================================================
# INIT
# =========================================================
dist_edge_endo = np.full(
    adataf.n_obs,
    np.nan
)

# =========================================================
# LOOP THROUGH SAMPLES
# =========================================================
for sample in adataf.obs["Sample"].unique():

    print("Processing:", sample)

    # -------------------------------------------------
    # SAMPLE MASK
    # -------------------------------------------------
    mask = (
        adataf.obs["Sample"] == sample
    )

    idx_in_adata = np.where(mask)[0]

    cells_in_sample = (
        adataf.obs_names[mask]
        .astype(str)
    )

    # -------------------------------------------------
    # PERICYTES IN SAMPLE
    # -------------------------------------------------
    peri_mask = (
        mask &
        (
            adataf.obs["merged_cell_type"]
            == "Mesenchymal Cells"
        )
    )

    peri_ids = (
        adataf.obs_names[peri_mask]
        .astype(str)
    )

    peri_polys = [
        cell_polygons[c]
        for c in peri_ids
        if c in cell_polygons
    ]

    if len(peri_polys) == 0:

        print(
            f"  No pericytes in {sample}. "
            f"Skipping."
        )

        continue

    # -------------------------------------------------
    # BUILD TREE
    # -------------------------------------------------
    tree = STRtree(peri_polys)

    peri_geoms_arr = np.array(
        peri_polys,
        dtype=object
    )

    # -------------------------------------------------
    # COMPUTE DISTANCES
    # -------------------------------------------------
    for i, cid in zip(
        idx_in_adata,
        cells_in_sample
    ):

        poly = cell_polygons.get(cid)

        if (
            poly is None or
            poly.is_empty
        ):
            continue

        nearest = tree.nearest(poly)

        if nearest is None:
            continue

        # shapely compatibility
        if hasattr(nearest, "geom_type"):

            nearest_geom = nearest

        else:

            nearest_geom = peri_geoms_arr[nearest]

        # -------------------------------------------------
        # TRUE EDGE-TO-EDGE DISTANCE
        # -------------------------------------------------
        d = poly.distance(nearest_geom)

        # convert pixels -> µm
        dist_edge_endo[i] = d * 0.108

# =========================================================
# STORE
# =========================================================
adataf.obs[
    "dist_edge_mesenchymal_um"
] = dist_edge_endo

print("\n✔ DONE")

# =========================================================
# SANITY CHECK
# =========================================================
print(
    adataf.obs[
        "dist_edge_mesenchymal_um"
    ].describe()
)

In [ ]:
import numpy as np
from shapely.strtree import STRtree

# =========================================================
# INIT
# =========================================================
dist_edge_endo = np.full(
    adataf.n_obs,
    np.nan
)

# =========================================================
# LOOP THROUGH SAMPLES
# =========================================================
for sample in adataf.obs["Sample"].unique():

    print("Processing:", sample)

    # -------------------------------------------------
    # SAMPLE MASK
    # -------------------------------------------------
    mask = (
        adataf.obs["Sample"] == sample
    )

    idx_in_adata = np.where(mask)[0]

    cells_in_sample = (
        adataf.obs_names[mask]
        .astype(str)
    )

    # -------------------------------------------------
    # PERICYTES IN SAMPLE
    # -------------------------------------------------
    peri_mask = (
        mask &
        (
            adataf.obs["merged_cell_type"]
            == "Beta Cells"
        )
    )

    peri_ids = (
        adataf.obs_names[peri_mask]
        .astype(str)
    )

    peri_polys = [
        cell_polygons[c]
        for c in peri_ids
        if c in cell_polygons
    ]

    if len(peri_polys) == 0:

        print(
            f"  No pericytes in {sample}. "
            f"Skipping."
        )

        continue

    # -------------------------------------------------
    # BUILD TREE
    # -------------------------------------------------
    tree = STRtree(peri_polys)

    peri_geoms_arr = np.array(
        peri_polys,
        dtype=object
    )

    # -------------------------------------------------
    # COMPUTE DISTANCES
    # -------------------------------------------------
    for i, cid in zip(
        idx_in_adata,
        cells_in_sample
    ):

        poly = cell_polygons.get(cid)

        if (
            poly is None or
            poly.is_empty
        ):
            continue

        nearest = tree.nearest(poly)

        if nearest is None:
            continue

        # shapely compatibility
        if hasattr(nearest, "geom_type"):

            nearest_geom = nearest

        else:

            nearest_geom = peri_geoms_arr[nearest]

        # -------------------------------------------------
        # TRUE EDGE-TO-EDGE DISTANCE
        # -------------------------------------------------
        d = poly.distance(nearest_geom)

        # convert pixels -> µm
        dist_edge_endo[i] = d * 0.108

# =========================================================
# STORE
# =========================================================
adataf.obs[
    "dist_edge_beta_um"
] = dist_edge_endo

print("\n✔ DONE")

# =========================================================
# SANITY CHECK
# =========================================================
print(
    adataf.obs[
        "dist_edge_beta_um"
    ].describe()
)

# SECTION 10 — SAVE ANNOTATED OBJECT
 Write fully annotated dataset with all distance columns
 This file is the input for all downstream figure notebooks


In [ ]:

adataf.write_h5ad('healthycells_allsamples_no9317162_labeledCV.h5ad')